
# Semana 7 · DATAPREP 360
## Control de calidad y validación de datos

**Curso:** Proyecto Productivo IB · Ciencia de Datos e Inteligencia Artificial  
**Producto de la sesión:** dataset validado + resumen de reglas + incidencias + métricas + plan de acción.

### Reto
El dataset de la Semana 6 ya fue limpiado y transformado. Ahora debemos demostrar, con reglas reproducibles, qué datos cumplen los criterios definidos, cuáles requieren revisión y qué incidencias impiden afirmar que el conjunto está listo para análisis.

> **Principio de trabajo:** validar no es volver a limpiar. Primero medimos, clasificamos y documentamos. No corregimos silenciosamente.



## 1. Archivo de entrada

Trabajaremos con el producto de la Semana 6:

`dataprep360_limpio_v1.csv`

La notebook tolera una columna accidental `Unnamed: 0` y verifica que exista el esquema mínimo antes de ejecutar las reglas.


In [4]:

# OPCIÓN A: Google Colab
# Descomenta estas líneas si necesitas subir el archivo manualmente.
#
# from google.colab import files
# uploaded = files.upload()
# ARCHIVO = next(iter(uploaded))

# OPCIÓN B: archivo en la misma carpeta de la notebook
ARCHIVO = "dataprep360_limpio_v1.csv"

print("Archivo de trabajo:", ARCHIVO)


Archivo de trabajo: dataprep360_limpio_v1.csv


In [7]:
# ============================================================
# SEMANA 7 - DATAPREP 360
# BLOQUE 2: IMPORTACIÓN Y CARGA DEL DATASET LIMPIO
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

# Configuración de visualización
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

# ------------------------------------------------------------
# 1. VERIFICAR SI EL ARCHIVO YA ESTÁ DISPONIBLE
# ------------------------------------------------------------

ARCHIVO = "dataprep360_limpio_v1.csv"

if not Path(ARCHIVO).is_file():

    print("El dataset limpio no se encuentra en el entorno.")
    print("Selecciona el archivo generado durante la Semana 6.")

    # Abrir el explorador para subir el archivo
    from google.colab import files

    uploaded = files.upload()

    # Identificar el archivo limpio cargado
    candidatos = [
        nombre for nombre in uploaded
        if nombre.lower().endswith(".csv")
        and "limpio_v1" in nombre.lower()
    ]

    if len(candidatos) != 1:
        raise ValueError(
            "Debes subir el archivo dataprep360_limpio_v1.csv. "
            "No utilices el dataset sucio original."
        )

    ARCHIVO = candidatos[0]

# ------------------------------------------------------------
# 2. CARGAR EL DATASET
# ------------------------------------------------------------

df = pd.read_csv(
    ARCHIVO,
    encoding="utf-8-sig"
)

# Eliminar una columna de índice exportada accidentalmente
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

# ------------------------------------------------------------
# 3. VERIFICAR LA CARGA
# ------------------------------------------------------------

print("Dataset cargado correctamente.")
print("Archivo utilizado:", ARCHIVO)

print("\nDimensiones del dataset:")
print(df.shape)

print("\nPrimeros registros:")
display(df.head(8))

print("\nColumnas disponibles:")
print(df.columns.tolist())

Dataset cargado correctamente.
Archivo utilizado: dataprep360_limpio_v1.csv

Dimensiones del dataset:
(730, 19)

Primeros registros:


,registro_id,fecha_evento,sede,categoria,canal,cantidad,valor_soles,duracion_min,estado,codigo_referencia,fuente,observacion,flag_duplicado_logico,flag_fecha_invalida,flag_cantidad_imposible,flag_valor_imposible,flag_duracion_imposible,flag_valor_atipico,flag_duracion_atipica
0,REG-0001,2026-03-17,Sur,Atención,Presencial,53.0,101.02,32.1,Cerrado,REF-6959,Sistema,Revisar documento,False,False,False,False,False,False,False
1,REG-0002,2026-07-21,Centro,Monitoreo,Teléfono,36.0,178.01,39.5,Cerrado,REF-9827,Importación,Revisar documento,False,False,False,False,False,False,False
2,REG-0003,2026-06-29,Centro,Monitoreo,Web,38.0,99.85,22.1,Pendiente,REF-4557,Formulario,Registro regular,False,False,False,False,False,False,False
3,REG-0004,2026-05-20,Centro,Incidencia,Teléfono,22.0,151.58,97.6,Cerrado,REF-4822,Sistema,Registro regular,False,False,False,False,False,False,True
4,REG-0005,2026-05-19,Norte,Monitoreo,App,78.0,76.61,11.5,En proceso,REF-6057,Sistema,Revisar documento,False,False,False,False,False,False,False
5,REG-0006,2026-08-05,Centro,Incidencia,Web,32.0,34.65,7.5,Cerrado,REF-4529,Sistema,Sin observación,False,False,False,False,False,False,False
6,REG-0007,2026-03-16,Sur,Incidencia,Presencial,69.0,186.49,96.6,Cerrado,REF-8062,Sistema,Sin observación,False,False,False,False,False,False,True
7,REG-0008,2026-07-07,Norte,Incidencia,Teléfono,35.0,292.63,9.0,Pendiente,REF-4852,Formulario,Confirmar dato,False,False,False,False,False,False,False



Columnas disponibles:
['registro_id', 'fecha_evento', 'sede', 'categoria', 'canal', 'cantidad', 'valor_soles', 'duracion_min', 'estado', 'codigo_referencia', 'fuente', 'observacion', 'flag_duplicado_logico', 'flag_fecha_invalida', 'flag_cantidad_imposible', 'flag_valor_imposible', 'flag_duracion_imposible', 'flag_valor_atipico', 'flag_duracion_atipica']



## 2. Contrato estructural del dataset

Antes de validar los valores, comprobamos que el archivo tenga las columnas indispensables.  
Esto permite **fallar temprano**: si falta una variable crítica, no tiene sentido calcular indicadores de calidad sobre un archivo incompleto.


In [ ]:

COLUMNAS_NUCLEO = [
    "registro_id", "fecha_evento", "sede", "categoria", "canal",
    "cantidad", "valor_soles", "duracion_min", "estado",
    "codigo_referencia", "fuente", "observacion"
]

faltan_nucleo = [c for c in COLUMNAS_NUCLEO if c not in df.columns]

if faltan_nucleo:
    raise ValueError(
        "El archivo no cumple el esquema esperado. "
        f"Faltan columnas: {faltan_nucleo}"
    )

print("✓ Esquema mínimo verificado.")



## 3. Preparación exclusiva para validar

Al volver a leer un CSV, una fecha se carga normalmente como texto. Convertirla a `datetime` aquí no es una nueva limpieza de negocio; es una preparación técnica necesaria para poder evaluar el periodo autorizado.

También garantizamos que las banderas generadas en la Semana 6 existan y sean booleanas.


Si faltan banderas del archivo limpio, se detiene el proceso: no se pueden inventar indicadores de trazabilidad.

In [ ]:
# Preparación para validar: comprobación de tipos y banderas heredadas.
fecha_original = df["fecha_evento"].copy()
df["fecha_evento"] = pd.to_datetime(
    df["fecha_evento"], format="%Y-%m-%d", errors="coerce"
)
fecha_no_convertida = fecha_original.notna() & df["fecha_evento"].isna()
if fecha_no_convertida.any():
    raise ValueError(
        f"Hay {int(fecha_no_convertida.sum())} fechas no interpretables en el CSV limpio. "
        "Revise la salida de la Semana 6 antes de validar."
    )

for col in ["cantidad", "valor_soles", "duracion_min"]:
    if not pd.api.types.is_numeric_dtype(df[col]):
        raise TypeError(
            f"La columna {col!r} debe ser numérica en el dataset limpio v1. "
            "Revise la conversión de la Semana 6."
        )

FLAGS_SEMANA6 = [
    "flag_fecha_invalida", "flag_cantidad_imposible",
    "flag_valor_imposible", "flag_duracion_imposible",
    "flag_duplicado_logico", "flag_valor_atipico",
    "flag_duracion_atipica",
]

faltan_flags = [c for c in FLAGS_SEMANA6 if c not in df.columns]
if faltan_flags:
    raise ValueError(
        f"Faltan banderas de trazabilidad de Semana 6: {faltan_flags}. "
        "No se reemplazan silenciosamente por False."
    )

mapa_booleano = {"true": True, "false": False, "1": True, "0": False}
for col in FLAGS_SEMANA6:
    if df[col].isna().any():
        raise ValueError(f"La bandera {col} contiene valores ausentes.")
    valores = df[col].astype("string").str.strip().str.lower()
    no_reconocidos = ~valores.isin(mapa_booleano)
    if no_reconocidos.any():
        raise ValueError(f"La bandera {col} contiene valores no booleanos.")
    df[col] = valores.map(mapa_booleano).astype(bool)

print("✓ Fechas, variables numéricas y banderas verificadas.")


## 4. Política de severidad del caso

| Severidad | Umbral | Interpretación |
|---|---:|---|
| **CRÍTICA** | 100% | Una falla puede bloquear el uso del dataset. |
| **ALTA** | 98% | Requiere corrección prioritaria. |
| **MEDIA** | 95% | Puede permitir uso condicionado, pero debe documentarse. |
| **AVISO** | Sin umbral | Señal de revisión; no implica borrar automáticamente. |

Los umbrales pertenecen al **caso DATAPREP 360**. En un proyecto real deben justificarse según el riesgo de la decisión.


## 5. Motor reusable de validación

In [ ]:

# Umbrales de aceptación por severidad.
UMBRALES = {
    "CRÍTICA": 100.0,
    "ALTA": 98.0,
    "MEDIA": 95.0,
}

resumen_reglas = []
metadatos_reglas = {}

def _serie_booleana(valor, indice):
    """Convierte una condición escalar o Series en Series alineada al DataFrame."""
    if isinstance(valor, pd.Series):
        return valor.reindex(indice)
    return pd.Series(bool(valor), index=indice)

def registrar_regla(
    rule_id,
    dimension,
    campo,
    descripcion,
    severidad,
    condicion_cumple,
    aplicable=None,
):
    """
    Ejecuta una regla y registra:
      - bandera de incumplimiento por fila,
      - cantidad de registros aplicables,
      - cantidad que cumple/incumple,
      - porcentaje de cumplimiento,
      - estado de la regla respecto al umbral.
    """
    indice = df.index

    cumple = _serie_booleana(condicion_cumple, indice).fillna(False).astype(bool)

    if aplicable is None:
        aplica = pd.Series(True, index=indice)
    else:
        aplica = _serie_booleana(aplicable, indice).fillna(False).astype(bool)

    incumple = aplica & (~cumple)
    col_error = f"err_{rule_id.lower()}"
    df[col_error] = incumple

    n_aplica = int(aplica.sum())
    n_incumple = int(incumple.sum())
    n_cumple = n_aplica - n_incumple

    porcentaje = (
        round(100 * n_cumple / n_aplica, 2)
        if n_aplica > 0 else np.nan
    )

    umbral = UMBRALES.get(severidad, np.nan)

    if n_aplica == 0:
        estado = "NO APLICA"
    else:
        estado = "CUMPLE" if porcentaje >= umbral else "NO CUMPLE"

    resumen_reglas.append({
        "regla": rule_id,
        "dimension": dimension,
        "campo": campo,
        "descripcion": descripcion,
        "severidad": severidad,
        "umbral_pct": umbral,
        "aplicables": n_aplica,
        "cumplen": n_cumple,
        "incumplen": n_incumple,
        "cumplimiento_pct": porcentaje,
        "estado_regla": estado,
    })

    metadatos_reglas[rule_id] = {
        "dimension": dimension,
        "campo": campo,
        "descripcion": descripcion,
        "severidad": severidad,
        "col_error": col_error,
    }



## 6. Reglas de completitud

En este caso distinguimos entre:
- campos **obligatorios** para identificar y contextualizar el evento;
- campos **analíticos** que deben estar presentes para ciertos análisis;
- campos **condicionales**, como `codigo_referencia`.


In [ ]:

# R01–R05: campos obligatorios principales
registrar_regla("R01", "Completitud", "registro_id",
                "Registro ID obligatorio", "CRÍTICA",
                df["registro_id"].notna())

registrar_regla("R02", "Completitud", "fecha_evento",
                "Fecha del evento obligatoria", "CRÍTICA",
                df["fecha_evento"].notna())

registrar_regla("R03", "Completitud", "sede",
                "Sede obligatoria", "ALTA",
                df["sede"].notna())

registrar_regla("R04", "Completitud", "categoria",
                "Categoría obligatoria", "ALTA",
                df["categoria"].notna())

registrar_regla("R05", "Completitud", "estado",
                "Estado obligatorio", "ALTA",
                df["estado"].notna())

# R06–R10: variables necesarias para análisis y trazabilidad
registrar_regla("R06", "Completitud", "canal",
                "Canal requerido para análisis operativo", "MEDIA",
                df["canal"].notna())

registrar_regla("R07", "Completitud", "cantidad",
                "Cantidad requerida para el análisis cuantitativo", "MEDIA",
                df["cantidad"].notna())

registrar_regla("R08", "Completitud", "valor_soles",
                "Valor monetario requerido para análisis económico", "MEDIA",
                df["valor_soles"].notna())

registrar_regla("R09", "Completitud", "duracion_min",
                "Duración requerida para análisis de tiempos", "MEDIA",
                df["duracion_min"].notna())

registrar_regla("R10", "Completitud", "fuente",
                "Fuente obligatoria para trazabilidad", "ALTA",
                df["fuente"].notna())

# R11: regla condicional del caso.
# Para Sistema o Importación exigimos código de referencia.
aplica_referencia = df["fuente"].isin(["Sistema", "Importación"])

registrar_regla("R11", "Completitud", "codigo_referencia",
                "Código de referencia requerido para Sistema o Importación",
                "MEDIA",
                df["codigo_referencia"].notna(),
                aplicable=aplica_referencia)

print("✓ Reglas R01–R11 ejecutadas.")


## 7. Unicidad

In [ ]:

id_presente = df["registro_id"].notna()
id_no_repetido = ~df["registro_id"].duplicated(keep=False)

registrar_regla(
    "R12", "Unicidad", "registro_id",
    "Registro ID no debe repetirse",
    "CRÍTICA",
    id_no_repetido,
    aplicable=id_presente
)

print("✓ Regla R12 ejecutada.")



## 8. Validez: patrón, dominio, periodo y rango

Una regla de validez traduce el significado de la variable a una condición verificable.


**Nota de trazabilidad:** cuando la Semana 6 sustituyó un valor imposible por NA, la regla de validez debe incluir su bandera histórica dentro de los registros aplicables. De lo contrario, un valor imposible desaparece del denominador y podría producir un 100 % engañoso.

In [ ]:

registrar_regla(
    "R13", "Validez", "registro_id",
    "Registro ID debe cumplir el patrón REG-0000",
    "ALTA",
    df["registro_id"].astype("string").str.fullmatch(r"REG-\d{4}", na=False),
    aplicable=df["registro_id"].notna()
)

registrar_regla(
    "R14", "Validez", "sede",
    "Sede debe pertenecer a {Centro, Norte, Sur}",
    "ALTA",
    df["sede"].isin({"Centro", "Norte", "Sur"}),
    aplicable=df["sede"].notna()
)

registrar_regla(
    "R15", "Validez", "categoria",
    "Categoría debe pertenecer al dominio permitido",
    "ALTA",
    df["categoria"].isin({"Atención", "Venta", "Incidencia", "Monitoreo"}),
    aplicable=df["categoria"].notna()
)

registrar_regla(
    "R16", "Validez", "canal",
    "Canal debe pertenecer al dominio permitido",
    "MEDIA",
    df["canal"].isin({"Teléfono", "Web", "Presencial", "App"}),
    aplicable=df["canal"].notna()
)

registrar_regla(
    "R17", "Validez", "estado",
    "Estado debe pertenecer a {Cerrado, Pendiente, En proceso}",
    "ALTA",
    df["estado"].isin({"Cerrado", "Pendiente", "En proceso"}),
    aplicable=df["estado"].notna()
)

registrar_regla(
    "R18", "Validez", "fuente",
    "Fuente debe pertenecer a {Sistema, Formulario, Importación}",
    "MEDIA",
    df["fuente"].isin({"Sistema", "Formulario", "Importación"}),
    aplicable=df["fuente"].notna()
)

fecha_en_periodo = df["fecha_evento"].between(
    pd.Timestamp("2026-03-01"),
    pd.Timestamp("2026-08-31")
)

registrar_regla(
    "R19", "Validez", "fecha_evento",
    "Fecha debe estar entre 01/03/2026 y 31/08/2026",
    "CRÍTICA",
    fecha_en_periodo & (~df["flag_fecha_invalida"]),
    aplicable=df["fecha_evento"].notna() | df["flag_fecha_invalida"]
)

registrar_regla(
    "R20", "Validez", "cantidad",
    "Cantidad debe estar entre 0 y 500",
    "ALTA",
    (df["cantidad"].between(0, 500)) & (~df["flag_cantidad_imposible"]),
    aplicable=df["cantidad"].notna() | df["flag_cantidad_imposible"]
)

registrar_regla(
    "R21", "Validez", "valor_soles",
    "Valor monetario debe ser mayor o igual que 0",
    "ALTA",
    (df["valor_soles"].ge(0)) & (~df["flag_valor_imposible"]),
    aplicable=df["valor_soles"].notna() | df["flag_valor_imposible"]
)

registrar_regla(
    "R22", "Validez", "duracion_min",
    "Duración debe estar entre 0 y 1440 minutos",
    "ALTA",
    (df["duracion_min"].between(0, 1440)) & (~df["flag_duracion_imposible"]),
    aplicable=df["duracion_min"].notna() | df["flag_duracion_imposible"]
)

print("✓ Reglas R13–R22 ejecutadas.")



## 9. Consistencia con la trazabilidad de la Semana 6

Aquí verificamos que los `flags` y el valor analítico cuenten la misma historia.

Ejemplo: si `flag_cantidad_imposible = True`, la Semana 6 debió conservar la evidencia en el flag y retirar el valor imposible de la columna analítica.


In [ ]:

registrar_regla(
    "R23", "Consistencia", "fecha_evento + flag_fecha_invalida",
    "Si flag_fecha_invalida=True, fecha_evento debe quedar nula",
    "CRÍTICA",
    (~df["flag_fecha_invalida"]) | df["fecha_evento"].isna()
)

registrar_regla(
    "R24", "Consistencia", "cantidad + flag_cantidad_imposible",
    "Si flag_cantidad_imposible=True, cantidad debe quedar nula",
    "CRÍTICA",
    (~df["flag_cantidad_imposible"]) | df["cantidad"].isna()
)

registrar_regla(
    "R25", "Consistencia", "valor_soles + flag_valor_imposible",
    "Si flag_valor_imposible=True, valor_soles debe quedar nulo",
    "CRÍTICA",
    (~df["flag_valor_imposible"]) | df["valor_soles"].isna()
)

registrar_regla(
    "R26", "Consistencia", "duracion_min + flag_duracion_imposible",
    "Si flag_duracion_imposible=True, duracion_min debe quedar nula",
    "CRÍTICA",
    (~df["flag_duracion_imposible"]) | df["duracion_min"].isna()
)

print("✓ Reglas R23–R26 ejecutadas.")



## 10. Avisos de revisión

Un posible duplicado lógico o un atípico no es automáticamente un error.  
Los tratamos como **avisos** para que una persona revise contexto o fuente de verdad.


In [ ]:

AVISOS = {
    "A01": {
        "dimension": "Revisión",
        "campo": "clave de negocio",
        "descripcion": "Posible duplicado lógico; revisar contexto antes de eliminar",
        "severidad": "AVISO",
        "flag": "flag_duplicado_logico",
    },
    "A02": {
        "dimension": "Revisión",
        "campo": "valor_soles",
        "descripcion": "Posible atípico en valor; contrastar evidencia",
        "severidad": "AVISO",
        "flag": "flag_valor_atipico",
    },
    "A03": {
        "dimension": "Revisión",
        "campo": "duracion_min",
        "descripcion": "Posible atípico en duración; contrastar evidencia",
        "severidad": "AVISO",
        "flag": "flag_duracion_atipica",
    },
}

resumen_reglas_df = pd.DataFrame(resumen_reglas)

display(
    resumen_reglas_df[
        ["regla", "dimension", "campo", "severidad",
         "aplicables", "incumplen", "cumplimiento_pct", "estado_regla"]
    ]
)


## 11. Clasificación del estado de calidad por registro

In [ ]:

columnas_criticas_altas = [
    meta["col_error"]
    for meta in metadatos_reglas.values()
    if meta["severidad"] in {"CRÍTICA", "ALTA"}
]

columnas_medias = [
    meta["col_error"]
    for meta in metadatos_reglas.values()
    if meta["severidad"] == "MEDIA"
]

columnas_avisos = [a["flag"] for a in AVISOS.values()]

tiene_falla_mayor = df[columnas_criticas_altas].any(axis=1)
tiene_observacion = (
    df[columnas_medias].any(axis=1)
    | df[columnas_avisos].any(axis=1)
)

df["estado_calidad"] = np.select(
    [tiene_falla_mayor, tiene_observacion],
    ["NO CONFORME", "OBSERVADO"],
    default="ACEPTADO",
)

todas_columnas_error = [
    meta["col_error"] for meta in metadatos_reglas.values()
]

df["n_incumplimientos"] = df[todas_columnas_error].sum(axis=1)
df["n_avisos"] = df[columnas_avisos].sum(axis=1)

display(df["estado_calidad"].value_counts(dropna=False).to_frame("registros"))


## 12. Tabla larga de incidencias

In [ ]:

def valor_observado(fila, campo):
    """Devuelve el valor cuando el campo corresponde a una sola columna."""
    if " + " in campo or campo == "clave de negocio":
        return ""
    return fila.get(campo, "")

incidencias = []

for idx, fila in df.iterrows():

    # Incumplimientos de reglas R01–R26
    for rule_id, meta in metadatos_reglas.items():
        if bool(fila[meta["col_error"]]):
            incidencias.append({
                "fila": idx,
                "registro_id": fila.get("registro_id", pd.NA),
                "regla": rule_id,
                "dimension": meta["dimension"],
                "campo": meta["campo"],
                "severidad": meta["severidad"],
                "descripcion": meta["descripcion"],
                "valor_observado": valor_observado(fila, meta["campo"]),
                "estado_calidad": fila["estado_calidad"],
            })

    # Avisos A01–A03
    for aviso_id, aviso in AVISOS.items():
        if bool(fila.get(aviso["flag"], False)):
            incidencias.append({
                "fila": idx,
                "registro_id": fila.get("registro_id", pd.NA),
                "regla": aviso_id,
                "dimension": aviso["dimension"],
                "campo": aviso["campo"],
                "severidad": aviso["severidad"],
                "descripcion": aviso["descripcion"],
                "valor_observado": valor_observado(fila, aviso["campo"]),
                "estado_calidad": fila["estado_calidad"],
            })

incidencias_df = pd.DataFrame(incidencias)

print("Incidencias y avisos registrados:", len(incidencias_df))
display(incidencias_df.head(20))


## 13. Métricas globales por dimensión

In [ ]:

# Completitud: celdas presentes / celdas requeridas
campos_requeridos = [
    "registro_id", "fecha_evento", "sede", "categoria", "estado"
]
completitud_pct = (
    df[campos_requeridos].notna().sum().sum()
    / (len(df) * len(campos_requeridos))
    * 100
)

# Unicidad de ID: IDs distintos / IDs presentes
ids_presentes = df["registro_id"].dropna()
unicidad_pct = (
    ids_presentes.nunique() / len(ids_presentes) * 100
    if len(ids_presentes) else np.nan
)

def metrica_dimension(nombre):
    sub = resumen_reglas_df[resumen_reglas_df["dimension"] == nombre]
    total_aplica = sub["aplicables"].sum()
    total_cumple = sub["cumplen"].sum()
    return (
        total_cumple / total_aplica * 100
        if total_aplica else np.nan
    )

validez_pct = metrica_dimension("Validez")
consistencia_pct = metrica_dimension("Consistencia")

metricas_calidad = pd.DataFrame({
    "metrica": [
        "Completitud campos obligatorios",
        "Unicidad de registro_id",
        "Validez",
        "Consistencia",
        "Registros aceptados",
        "Registros observados",
        "Registros no conformes",
    ],
    "valor_pct": [
        completitud_pct,
        unicidad_pct,
        validez_pct,
        consistencia_pct,
        (df["estado_calidad"].eq("ACEPTADO").mean() * 100),
        (df["estado_calidad"].eq("OBSERVADO").mean() * 100),
        (df["estado_calidad"].eq("NO CONFORME").mean() * 100),
    ]
})

metricas_calidad["valor_pct"] = metricas_calidad["valor_pct"].round(2)

display(metricas_calidad)



## 14. Plan de acción

El reporte no debe terminar en “hay errores”. Debe indicar qué hacer con ellos.  
La acción depende de la severidad y, cuando corresponde, de una fuente de verdad externa.


In [ ]:

conteo_severidad = (
    incidencias_df["severidad"].value_counts()
    if not incidencias_df.empty
    else pd.Series(dtype="int64")
)

plan_accion = pd.DataFrame([
    {
        "severidad": "CRÍTICA",
        "incidencias": int(conteo_severidad.get("CRÍTICA", 0)),
        "accion": "Contrastar con fuente original. Corregir si existe evidencia; de lo contrario, excluir del análisis que dependa del campo y documentar.",
        "prioridad": "Inmediata",
    },
    {
        "severidad": "ALTA",
        "incidencias": int(conteo_severidad.get("ALTA", 0)),
        "accion": "Corregir prioritariamente o limitar el uso del registro hasta resolver la incidencia.",
        "prioridad": "Alta",
    },
    {
        "severidad": "MEDIA",
        "incidencias": int(conteo_severidad.get("MEDIA", 0)),
        "accion": "Revisar, documentar y decidir si el análisis puede continuar con la limitación explícita.",
        "prioridad": "Programada",
    },
    {
        "severidad": "AVISO",
        "incidencias": int(conteo_severidad.get("AVISO", 0)),
        "accion": "Revisar contexto. No eliminar automáticamente duplicados lógicos ni valores atípicos.",
        "prioridad": "Revisión",
    },
])

display(plan_accion)



## 15. Exportar evidencia reproducible

Generaremos:

1. `dataprep360_validado_semana7.csv`  
2. `incidencias_validacion_semana7.csv`  
3. `reporte_validacion_semana7.xlsx`

El Excel contiene cinco hojas: resumen de reglas, métricas, incidencias, dataset validado y plan de acción.


In [ ]:

ARCHIVO_VALIDADO = "dataprep360_validado_semana7.csv"
ARCHIVO_INCIDENCIAS = "incidencias_validacion_semana7.csv"
ARCHIVO_REPORTE = "reporte_validacion_semana7.xlsx"

df.to_csv(ARCHIVO_VALIDADO, index=False, encoding="utf-8-sig")
incidencias_df.to_csv(ARCHIVO_INCIDENCIAS, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(ARCHIVO_REPORTE, engine="openpyxl") as writer:
    resumen_reglas_df.to_excel(writer, sheet_name="Resumen_Reglas", index=False)
    metricas_calidad.to_excel(writer, sheet_name="Metricas_Calidad", index=False)
    incidencias_df.to_excel(writer, sheet_name="Incidencias", index=False)
    df.to_excel(writer, sheet_name="Dataset_Validado", index=False)
    plan_accion.to_excel(writer, sheet_name="Plan_Accion", index=False)

print("Archivos generados:")
print(" -", ARCHIVO_VALIDADO)
print(" -", ARCHIVO_INCIDENCIAS)
print(" -", ARCHIVO_REPORTE)



## 16. Decisión final del caso

No debemos resumir la calidad con un solo promedio. Una falla crítica puede ser importante aunque el porcentaje global sea alto.

La decisión se obtiene revisando:
- reglas **CRÍTICAS**;
- reglas **ALTAS**;
- porcentaje de registros **NO CONFORMES**;
- incidencias que requieren fuente de verdad;
- avisos que necesitan juicio de dominio.


In [ ]:

fallas_criticas = resumen_reglas_df[
    (resumen_reglas_df["severidad"] == "CRÍTICA")
    & (resumen_reglas_df["estado_regla"] == "NO CUMPLE")
]

fallas_altas = resumen_reglas_df[
    (resumen_reglas_df["severidad"] == "ALTA")
    & (resumen_reglas_df["estado_regla"] == "NO CUMPLE")
]

if len(fallas_criticas) > 0:
    estado_dataset = "REQUIERE CORRECCIÓN ANTES DEL ANÁLISIS"
elif len(fallas_altas) > 0:
    estado_dataset = "APTO SOLO CON CORRECCIONES PRIORITARIAS"
elif (df["estado_calidad"] == "OBSERVADO").any():
    estado_dataset = "APTO CON OBSERVACIONES DOCUMENTADAS"
else:
    estado_dataset = "APTO SEGÚN LAS REGLAS INTERNAS DEFINIDAS"

print("ESTADO DEL DATASET:")
print(estado_dataset)

print("\\nImportante:")
print("Este estado evalúa calidad interna. No demuestra exactitud frente a la realidad sin una fuente de verdad externa.")



## 17. Transferencia al proyecto de cada equipo

No copien las reglas de DATAPREP 360. Copien el **patrón**:

**REQUISITO → REGLA → SEVERIDAD → PRUEBA → INCIDENCIA → DECISIÓN → EVIDENCIA**

Cada equipo debe definir como mínimo:
- 5 reglas de completitud/validez;
- 1 regla de unicidad;
- 1 regla de consistencia;
- severidad justificada;
- evidencia que aún no puede comprobarse;
- plan de acción para incidencias.

### Pregunta de cierre
**¿Puedo defender técnicamente por qué mi dataset puede —o no puede— continuar hacia el análisis?**
